In [1]:
# ─────────────────────────────────────────
# 셀 6: 피드백 출력 확인
# ─────────────────────────────────────────
print("【 피드백 생성 테스트 】\n")

test_sent = "저는 항상 열심히 노력했으며 다양한 경험을 통해 깨달았습니다."
result = detect_hedge_expressions(test_sent)

print(f"입력: {test_sent}\n")
print(f"점수: {result['score']}점 ({result['grade']}등급)")
print(f"요약: {result['summary']}\n")

if result['feedback_items']:
    print("개선 제안:")
    for fb in result['feedback_items']:
        print(f"  [{fb['category']}] '{fb['original']}' {fb['suggestion']}")
else:
    print("피드백 없음 (헤지 표현 미탐지)")

【 피드백 생성 테스트 】



NameError: name 'detect_hedge_expressions' is not defined

In [6]:
# ─────────────────────────────────────────
# 셀 5: FN 분석 — 놓친 표현 목록
# ─────────────────────────────────────────
print("【 놓친 표현(FN) 목록 — 사전 보강 대상 】\n")

for i, (sent, gold) in enumerate(DUMMY_DATA, 1):
    result = detect_hedge_expressions(sent)
    detected = [h["term"] for h in result["hits"]]
    fn = [g for g in gold if g not in detected]
    
    if fn:
        print(f"  문장 {i}: {fn}  →  사전에 추가하세요")

print("\n【 오탐(FP) 목록 — 사전에서 제거 대상 】\n")

for i, (sent, gold) in enumerate(DUMMY_DATA, 1):
    result = detect_hedge_expressions(sent)
    detected = [h["term"] for h in result["hits"]]
    fp = [d for d in detected if not any(d in g or g in d for g in gold)]
    
    if fp:
        print(f"  문장 {i}: {fp}  →  사전에서 제거하세요")

【 놓친 표현(FN) 목록 — 사전 보강 대상 】

  문장 1: ['최선을']  →  사전에 추가하세요

【 오탐(FP) 목록 — 사전에서 제거 대상 】



In [5]:
# ─────────────────────────────────────────
# 셀 4: 재현율·정밀도·F1 계산
# ─────────────────────────────────────────
total_tp, total_fn, total_fp = 0, 0, 0

for sent, gold in DUMMY_DATA:
    result = detect_hedge_expressions(sent)
    detected = [h["term"] for h in result["hits"]]

    tp = [g for g in gold    if any(g in d or d in g for d in detected)]
    fn = [g for g in gold    if g not in detected]
    fp = [d for d in detected if not any(d in g or g in d for g in gold)]

    total_tp += len(tp)
    total_fn += len(fn)
    total_fp += len(fp)

recall    = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0
precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0
f1        = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print("="*40)
print(f"  TP (맞게 탐지)     : {total_tp}개")
print(f"  FN (놓친 표현)     : {total_fn}개  ← 사전에 추가 필요")
print(f"  FP (오탐)          : {total_fp}개  ← 사전에서 제거 필요")
print("─"*40)
print(f"  재현율  (Recall)   : {recall:.1%}")
print(f"  정밀도  (Precision): {precision:.1%}")
print(f"  F1 Score           : {f1:.1%}")
print("="*40)

  TP (맞게 탐지)     : 19개
  FN (놓친 표현)     : 1개  ← 사전에 추가 필요
  FP (오탐)          : 0개  ← 사전에서 제거 필요
────────────────────────────────────────
  재현율  (Recall)   : 95.0%
  정밀도  (Precision): 100.0%
  F1 Score           : 97.4%


In [4]:
# ─────────────────────────────────────────
# 셀 3: 문장별 탐지 결과 출력
# ─────────────────────────────────────────
for i, (sent, gold) in enumerate(DUMMY_DATA, 1):
    result = detect_hedge_expressions(sent)
    detected = [h["term"] for h in result["hits"]]
    
    print(f"\n{'='*55}")
    print(f"[문장 {i}] {sent}")
    print(f"  정답 레이블 : {gold}")
    print(f"  탐지된 표현 : {detected}")
    print(f"  모호도 점수 : {result['score']}점  ({result['grade']}등급) — {result['summary']}")
    print(f"  카테고리별  : {result['by_category']}")


[문장 1] 저는 항상 열심히 노력했으며 최선을 다해 업무에 임했습니다.
  정답 레이블 : ['열심히', '노력했으며', '최선을']
  탐지된 표현 : ['열심히', '노력했으며', '최선을 다해']
  모호도 점수 : 85점  (A등급) — 표현이 구체적이고 명확합니다.
  카테고리별  : {'A_effort': 3, 'B_quantifier': 0, 'C_abstract': 0, 'D_emotion': 0, 'E_uncertainty': 0}

[문장 2] 3개월간 A/B 테스트를 12회 진행해 전환율을 23% 개선했습니다.
  정답 레이블 : []
  탐지된 표현 : []
  모호도 점수 : 100점  (A등급) — 표현이 구체적이고 명확합니다.
  카테고리별  : {'A_effort': 0, 'B_quantifier': 0, 'C_abstract': 0, 'D_emotion': 0, 'E_uncertainty': 0}

[문장 3] 다양한 경험을 통해 소통 능력과 리더십을 키울 수 있었습니다.
  정답 레이블 : ['다양한', '소통 능력', '리더십', '키울 수 있었습니다']
  탐지된 표현 : ['다양한', '소통 능력', '리더십', '키울 수 있었습니다']
  모호도 점수 : 79점  (B등급) — 일부 모호 표현이 있으나 전반적으로 양호합니다.
  카테고리별  : {'A_effort': 0, 'B_quantifier': 1, 'C_abstract': 2, 'D_emotion': 0, 'E_uncertainty': 1}

[문장 4] 팀장으로서 책임감을 갖고 열정적으로 프로젝트를 이끌었습니다.
  정답 레이블 : ['책임감', '열정적으로']
  탐지된 표현 : ['책임감', '열정적으로']
  모호도 점수 : 85점  (A등급) — 표현이 구체적이고 명확합니다.
  카테고리별  : {'A_effort': 1, 'B_quantifier': 0, 'C_abstract': 1, 'D_emotion': 0, 'E_uncertainty': 0}


In [ ]:
# ─────────────────────────────────────────
# 셀 2: 더미 문장 + 정답 레이블 정의
# ─────────────────────────────────────────
DUMMY_DATA = [
    (
        "저는 항상 열심히 노력했으며 최선을 다해 업무에 임했습니다.",
        ["열심히", "노력했으며", "최선을"]
    ),
    (
        "3개월간 A/B 테스트를 12회 진행해 전환율을 23% 개선했습니다.",
        []
    ),
    (
        "다양한 경험을 통해 소통 능력과 리더십을 키울 수 있었습니다.",
        ["다양한", "소통 능력", "리더십", "키울 수 있었습니다"]
    ),
    (
        "팀장으로서 책임감을 갖고 열정적으로 프로젝트를 이끌었습니다.",
        ["책임감", "열정적으로"]
    ),
    (
        "이 경험을 통해 협업의 중요성을 깨달았습니다.",
        ["깨달았습니다"]
    ),
    (
        "Python과 SQL을 활용해 고객 이탈률을 15%p 낮췄습니다.",
        []
    ),
    (
        "다수의 프로젝트에서 적극적으로 참여한 것 같습니다.",
        ["다수의", "적극적으로", "것 같습니다"]
    ),
    (
        "매주 코드 리뷰를 주도해 팀 버그 발생률을 40% 줄였습니다.",
        []
    ),
    (
        "성실하게 자기개발에 힘쓰며 여러 역량을 쌓았습니다.",
        ["성실하게", "x자기개발", "여러", "역량"]
    ),
    (
        "도전정신으로 어려운 문제도 포기하지 않고 최선을 다합니다.",
        ["도전정신", "최선을"]
    ),
]

print(f"더미 문장 총 {len(DUMMY_DATA)}개 로드 완료")

더미 문장 총 10개 로드 완료


In [2]:
# ─────────────────────────────────────────
# 셀 1: 라이브러리 & 모듈 import
# ─────────────────────────────────────────
import sys
import os

sys.path.append('../../models/role03_hedge')

from hedge_detector import detect_hedge_expressions

# 확인용
print(os.listdir('../../models/role03_hedge'))

['.gitkeep', 'abstract_dict_v1.py', 'hedge_detector.py', 'hedge_dict_v1.py', '__pycache__']
